<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula03b%20-%20gridsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Aula 03b: Otimização de Hiperparâmetros (GridSearchCV) e Validação Cruzada Aninhada

Seja muito bem-vindo(a) ao ápice prático da construção de modelos de Machine Learning!

---

### 🎯 Qual é o objetivo desta aula?
Até agora, quando usávamos o algoritmo k-NN, nós escolhíamos o valor de $k$ meio 'no palpite' ($k=5$ ou $k=15$).  
Mas e se o melhor valor fosse $k=3$, ou $k=7$, ou $k=9$?  
E mais: o k-NN tem outros botões para girar, como:
* O tipo de peso dos vizinhos (`weights='uniform'` ou `weights='distance'`);
* A métrica de distância ($p=1$ para Manhattan ou $p=2$ para Pitágoras/Euclidiana).

Nesta aula, o professor Boldt nos ensina como **automatizar completamente a busca pela melhor combinação de hiperparâmetros** sem se enganar e sem vazar dados, culminando no padrão-ouro da literatura científica: a **Validação Cruzada Aninhada (Nested Cross-Validation)**!

---

### 🧭 A Trilha Pedagógica do Professor Boldt
1. **Diferença Conceitual**: Parâmetros (o modelo aprende sozinho, como os pesos $w$) vs Hiperparâmetros (nós escolhemos antes, como o $k$).
2. **Abordagem Ingênua: Conjunto de Validação (Treino, Validação e Teste)**:
   * Treino para ajustar o modelo, Validação para escolher o melhor $k$, e Teste para a avaliação final honesta.
3. **O Erro Sutil: *"Não é a melhor forma. Não mesmo!"***:
   * Por que calcular o `scaler` antes da divisão interna contamina a validação.
4. **Otimização via Validação Cruzada Interna**:
   * Substituir um único conjunto de validação por um K-Fold interno.
5. **A Fusão Perfeita: `Pipeline` + `GridSearchCV`**:
   * Como criar esteiras de automação e passar parâmetros com a sintaxe `nome_da_etapa__parametro`.
6. **O Padrão Ouro Científico: Validação Cruzada Aninhada (Nested CV)**:
   * Rodar um `cross_validate` por fora de um `GridSearchCV` por dentro para medir a capacidade real de generalização da solução!



## 1. Ponto de Partida: Divisão Inicial em Treino e Teste Externo

> O conjunto de teste (20%) é o nosso cofre blindado. Ele só pode ser tocado no final para reportar o resultado real do modelo aos clientes/pesquisadores.


In [178]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Carregamos os dados
X, y = load_wine(return_X_y=True)
# Separamos 20% para o teste definitivo final
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Padronizamos
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo de base com k=5
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Acurácia de referência (k=5): {accuracy_score(y_test, y_pred) * 100:.2f}%")


accuracy: 0.9444444444444444


## 2. Abordagem 1: Dividindo o Treino em Treino2 e Validação (Busca Manual por k)

> **A Estratégia dos Três Conjuntos**:  
> * **`X_train2`**: Usado para ajustar os vizinhos.  
> * **`X_val` (Validação Interna)**: Usado para testar cada $k \in [1, 3, 5, 7, 9, 11, 13, 15]$ e eleger o campeão.  
> * **`X_test` (Validação Externa)**: Usado para dar a nota final do campeão.


In [183]:
# Dividimos o X_train original em X_train2 (treino interno) e X_val (validação interna)
X_train2, X_val, y_train2, y_val = train_test_split(X_train, y_train, test_size=0.2)

k_values = [1, 3, 5, 7, 9, 11, 13, 15]
best_accuracy = 0
best_k = 0

# --- 1. VALIDAÇÃO INTERNA: Escolha do melhor k ---
for k in k_values:
  model = KNeighborsClassifier(n_neighbors=k)
  X_train2_scaled = scaler.transform(X_train2)
  X_val_scaled = scaler.transform(X_val)
  model.fit(X_train2_scaled, y_train2)
  y_pred = model.predict(X_val_scaled)
  acc = accuracy_score(y_val, y_pred)
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"Melhor k escolhido na validação: {best_k}")
print(f"Melhor acurácia na validação: {best_accuracy * 100:.2f}%")

# --- 2. VALIDAÇÃO EXTERNA: Teste final do campeão ---
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train) # Retreinamos no X_train completo
y_pred = best_model.predict(X_test_scaled)
print(f"Acurácia no Teste Final: {accuracy_score(y_test, y_pred) * 100:.2f}%")


best k: 3
best accuracy: 1.0
test accuracy: 0.9444444444444444


### O Alerta do Professor: *'Não é a melhor forma. Não mesmo!'*
Na célula abaixo, o professor demonstra o erro que milhares de iniciantes cometem sem perceber:  
Ele calculou `scaler.transform(X_train)` **antes** de separar `X_val`!  
Isso significa que as amostras de validação influenciaram a média e o desvio padrão da escala, ocorrendo contaminação entre treino e validação.

In [189]:
### Não é a melhor forma. Não mesmo! (Escalou tudo antes de dividir)
X_train_scaled = scaler.transform(X_train)
X_train2, X_val, y_train2, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2)

# Validação Interna
best_accuracy = 0
best_k = 0
for k in k_values:
  model = KNeighborsClassifier(n_neighbors=k)
  model.fit(X_train2, y_train2)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"Melhor k: {best_k}")
print(f"Acurácia de validação (contaminada): {best_accuracy * 100:.2f}%")

# Validação Externa
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"Acurácia no teste: {accuracy_score(y_test, y_pred) * 100:.2f}%")


best k: 3
best accuracy: 1.0
test accuracy: 0.9444444444444444


## 3. Abordagem 2: Escolhendo o Hiperparâmetro com Validação Cruzada Interna

> Em vez de depender de um único conjunto de validação (que pode ser pequeno e ruidoso), usamos **Validação Cruzada (5-Fold)** dentro do conjunto de treino para escolher o melhor $k$!

In [190]:
# A mesma coisa, mas com validação cruzada
from sklearn.model_selection import cross_validate, KFold

# Validação interna com KFold embaralhado
spliter = KFold(n_splits=5, shuffle=True, random_state=42)
best_accuracy = 0
best_k = 0

for k in k_values:
  model = KNeighborsClassifier(n_neighbors=k)
  # Avalia a média dos 5 folds para este k
  scores = cross_validate(model, X_train_scaled, y_train, cv=spliter)
  acc = scores['test_score'].mean()
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"Melhor k via Cross-Validation: {best_k}")
print(f"Acurácia média interna: {best_accuracy * 100:.2f}%")

# Validação externa definitiva
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"Acurácia no teste final: {accuracy_score(y_test, y_pred) * 100:.2f}%")


best k: 9
best accuracy: 0.97192118226601
test accuracy: 0.9444444444444444


### *'Assim é a melhor forma! Com pipeline fica blindado'*
Aqui unimos a Validação Cruzada com o `make_pipeline`. O scaler é recalculado internamente para cada um dos 5 folds sem vazar nada!

In [192]:
# Assim é a melhor forma! Com pipeline fica 100% blindado contra vazamento
from sklearn.pipeline import make_pipeline

best_accuracy = 0
best_k = 0

for k in k_values:
  # O Pipeline recalcula o StandardScaler em cada fold automaticamente
  model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
  scores = cross_validate(model, X_train, y_train, cv=spliter)
  acc = scores['test_score'].mean()
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"Melhor k (com Pipeline): {best_k}")
print(f"Acurácia média de validação: {best_accuracy * 100:.2f}%")

# Validação externa
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"Acurácia no teste final: {accuracy_score(y_test, y_pred) * 100:.2f}%")


best k: 15
best accuracy: 0.9719211822660098
test accuracy: 0.9722222222222222


## 4. Automatizando a Busca com `GridSearchCV`

> **O que é o `GridSearchCV`?**  
> Ele substitui todo aquele loop `for k in k_values` por uma única chamada profissional!  
> Você define um dicionário `param_grid` com os valores que quer testar e o `GridSearchCV` cuida de todo o treino, validação cruzada e seleção do melhor modelo.

In [198]:
from sklearn.model_selection import GridSearchCV

model = KNeighborsClassifier()
# Dicionário com a grade de parâmetros a serem testados
param_grid = {'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}

# GridSearchCV automatiza o teste de cada valor com o divisor KFold
grid_search = GridSearchCV(model, param_grid, cv=spliter)
grid_search.fit(X_train_scaled, y_train)

# Extraímos o melhor modelo encontrado
best_model = grid_search.best_estimator_
print(f"Melhor k encontrado: {best_model.get_params()['n_neighbors']}")
print(f"Melhor acurácia interna (best_score_): {grid_search.best_score_ * 100:.2f}%")

# Validação externa
y_pred = best_model.predict(X_test_scaled)
print(f"Acurácia no teste final: {accuracy_score(y_test, y_pred) * 100:.2f}%")
### Mas assim ainda escalou fora do split... Como juntar Pipeline com GridSearchCV?


best k: 3
best accuracy: 0.9857142857142858
test accuracy: 0.9444444444444444


### Como Usar `GridSearchCV` Junto com `Pipeline` (Sintaxe `nome__parametro`)
Para passar parâmetros para um modelo que está dentro de um Pipeline, usamos o nome da etapa seguido de dois underlines `__`:  
`'model__n_neighbors': [1, 3, 5, ...]`

In [205]:
from sklearn.pipeline import Pipeline

# Pipeline com etapas nomeadas ('scaler' e 'model')
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

# Atenção à sintaxe: 'model__n_neighbors' avisa ao GridSearch que o parâmetro pertence à etapa 'model'
param_grid = {'model__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}

grid_search = GridSearchCV(pipe, param_grid, cv=spliter)
# Passamos os dados de treino PUROS (X_train)! O scaler é executado dentro de cada fold!
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Melhor k: {best_model.get_params()['model__n_neighbors']}")
print(f"Melhor acurácia interna: {grid_search.best_score_ * 100:.2f}%")

# Validação externa: o pipeline transforma X_test automaticamente!
y_pred = best_model.predict(X_test)
print(f"Acurácia no Teste Final: {accuracy_score(y_test, y_pred) * 100:.2f}%")
### Assim é a melhor forma de treinar e testar!


best k: 9
best accuracy: 0.972167487684729
test accuracy: 0.9444444444444444


## 5. O Padrão Ouro: Validação Cruzada Aninhada (Nested Cross-Validation)

> **O que é Validação Cruzada Aninhada?**  
> `scores = cross_validate(grid_search, X, y, cv=spliter)`  
> * **Loop Externo (`cross_validate`)**: Simula a chegada de dados de teste novos e independentes.  
> * **Loop Interno (`grid_search`)**: Em cada rodada do loop externo, faz outra validação cruzada para achar o melhor $k$.  
> **Por que isso é tão importante?**  
> Porque o `best_score_` do GridSearch pode ser ligeiramente otimista (ele testou muitas opções e pegou a maior nota). O Nested CV dá a estimativa **100% imparcial e científica** do desempenho de todo o processo de Machine Learning!

In [211]:
# Avaliação imparcial de todo o processo via Nested CV
scores = cross_validate(grid_search, X, y, cv=spliter)
print(f"Acurácia estimada via Nested Cross-Validation: {scores['test_score'].mean() * 100:.2f}%")


accuracy: 0.9665079365079364


In [213]:
# Com verbose=1 o GridSearchCV imprime o progresso de cada teste
grid_search = GridSearchCV(pipe, param_grid, cv=spliter, verbose=1)
scores = cross_validate(grid_search, X, y, cv=spliter)
print(f"Acurácia Nested CV: {scores['test_score'].mean() * 100:.2f}%")


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
accuracy: 0.9547619047619047


### Otimização Simultânea de Múltiplos Hiperparâmetros
O professor fecha o curso combinando 3 hiperparâmetros ao mesmo tempo:  
1. `n_neighbors`: 8 valores possíveis (1 a 15).  
2. `weights`: 'uniform' (todos os vizinhos têm o mesmo peso) ou 'distance' (vizinhos mais pertinho têm voto com maior peso).  
3. `p`: 1 (Distância Manhattan - quadras de cidade) ou 2 (Distância Euclidiana - linha reta).  
Total: $8 \times 2 \times 2 = 32$ configurações avaliadas com o máximo rigor metodológico!

In [214]:
# Grade rica de hiperparâmetros (32 combinações)
param_grid = {
    'model__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}

grid_search = GridSearchCV(pipe, param_grid, cv=spliter, verbose=1)
scores = cross_validate(grid_search, X, y, cv=spliter)
print(f"Acurácia Final do Modelo Otimizado (Nested CV): {scores['test_score'].mean() * 100:.2f}%")


Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
accuracy: 0.9606349206349206


---
## 🎓 Resumo da Aula 03b & Fechamento do Curso:
1. **Hiperparâmetros**: Configurações que governam o aprendizado do modelo e devem ser testadas sistematicamente.
2. **Conjunto de Validação**: Necessário para calibrar hiperparâmetros sem contaminar o teste final.
3. **GridSearchCV**: A ferramenta de busca em grade exaustiva do Scikit-Learn.
4. **Pipeline + GridSearchCV**: A combinação perfeita (`nome__hiperparametro`) que impede contaminação de pré-processamento.
5. **Nested Cross-Validation**: A técnica definitiva de avaliação de algoritmos que garante que sua inteligência artificial funcionará no mundo real com a mesma precisão observada no laboratório!

